# Grammar–KT pipeline walkthrough

This notebook mirrors `scripts/run.py`: scientific declarations are loaded directly, then passed to chronological stage functions. There is no experiment configuration layer.

`Typed EGP resource → normalisation → canonicalisation → generation → validation → semantic grammar fold → development acquisition and frozen probes → KC selection → projection → KT → evaluation`

In [1]:
import json, sys
from copy import deepcopy
from functools import partial
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / 'modules').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from grammar_kt.canonicalise import canonicalise
from grammar_kt.evaluate import evaluate
from grammar_kt.fold import build_semantic_fold
from grammar_kt.generate import generate_items
from grammar_kt.io import call_model, load_typed_resource, read_text, read_yaml
from grammar_kt.kc import project_kcs
from grammar_kt.kc_candidates import make_kc_candidates
from grammar_kt.kc_selection import select_kcs
from grammar_kt.kt import run_kt
from grammar_kt.normalise import normalise
from grammar_kt.simulate import materialize_latent_world, simulate_frozen_probes
from grammar_kt.validate_items import bank_summary, select_item_bank, validate_items

LIVE_MODE = False
NOTEBOOK_LEARNERS = 8
temporary_run = TemporaryDirectory(prefix='grammar_kt_walkthrough_')
WORK = Path(temporary_run.name)
print('Fixture mode: deterministic model responses' if not LIVE_MODE else 'Live model mode')

Fixture mode: deterministic model responses


/usr/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Research declarations

Every file is named and loaded here. The variables below are the actual scientific objects consumed by the stages.

In [2]:
RESOURCE_PATH = ROOT / 'data/fixtures/egp_pilot.jsonl'
resource_schema = read_yaml(ROOT / 'modules/grammar/resource/egp/schema.yaml')

phase1_prompt = read_text(ROOT / 'modules/grammar/resource/egp/normalisation/phase1.txt')
phase2_prompt = read_text(ROOT / 'modules/grammar/resource/egp/normalisation/phase2.txt')
normalisation_rulebook = read_text(ROOT / 'modules/grammar/resource/egp/normalisation/rulebook.md')
grammar_schema = read_yaml(ROOT / 'modules/grammar/canonical/schema.yaml')
operation_design = read_yaml(ROOT / 'modules/grammar/canonical/english_operations.yaml')

generation_prompt = read_text(ROOT / 'modules/items/generation/prompt.txt')
generation_rulebook = read_text(ROOT / 'modules/items/generation/rulebook.md')
active_generation_design = read_yaml(ROOT / 'modules/items/generation/design.yaml')
fixture_generation_design = read_yaml(ROOT / 'data/fixtures/item_generation.yaml')
generation_design = active_generation_design if LIVE_MODE else fixture_generation_design
item_format = read_yaml(ROOT / 'modules/items/generation/formats/controlled_production.yaml')

validation_prompt = read_text(ROOT / 'modules/items/validation/prompt.txt')
validation_criteria = read_yaml(ROOT / 'modules/items/validation/criteria.yaml')
active_backends = read_yaml(ROOT / 'modules/model_backends.yaml')

grammar_fold_design = read_yaml(ROOT / 'data/fixtures/semantic_fold.yaml')
simulation_world_design = read_yaml(ROOT / 'data/fixtures/declarations/simulation_world.yaml')
simulation_protocol = read_yaml(ROOT / 'modules/simulation/protocol.yaml')

candidate_design = read_yaml(ROOT / 'modules/kcs/candidate_design.yaml') | {
    'operation_declarations': operation_design['operations']
}
selection_design = read_yaml(ROOT / 'modules/kcs/selection.yaml')

kt_protocol = read_yaml(ROOT / 'modules/evaluation/kt/protocol.yaml')
evaluation_protocol = read_yaml(ROOT / 'modules/evaluation/protocol.yaml')

if LIVE_MODE:
    model_call = call_model
    backend_settings = active_backends
else:
    model_call = partial(
        call_model,
        fixture_responses=read_yaml(ROOT / 'data/fixtures/model_responses.yaml'),
    )
    backend_settings = {
        stage: {'model': 'fixture', 'reasoning_effort': 'deterministic'}
        for stage in ('normalisation', 'generation', 'validation')
    }

display(pd.DataFrame([
    {'stage': 'resource', 'declaration': resource_schema['resource_id']},
    {'stage': 'normalisation', 'declaration': phase1_prompt.splitlines()[0] + ' / ' + phase2_prompt.splitlines()[0]},
    {'stage': 'canonicalisation', 'declaration': grammar_schema['schema_id']},
    {'stage': 'generation', 'declaration': generation_design['design_id'] + ' / ' + item_format['format_id']},
    {'stage': 'validation', 'declaration': validation_criteria['policy_id']},
    *[{'stage': f'{stage} backend', 'declaration': f"{settings['model']} / {settings['reasoning_effort']}"}
      for stage, settings in backend_settings.items()],
    {'stage': 'grammar fold', 'declaration': grammar_fold_design['fold_id']},
    {'stage': 'simulation', 'declaration': simulation_world_design['world_id'] + ' / ' + simulation_protocol['protocol_id']},
    {'stage': 'KC candidates', 'declaration': candidate_design['candidate_design_id']},
    {'stage': 'KC selection', 'declaration': selection_design['selection_id']},
    {'stage': 'KT', 'declaration': kt_protocol['protocol_id']},
    {'stage': 'evaluation', 'declaration': evaluation_protocol['protocol_id']},
]))

,stage,declaration
0,resource,egp_english_pilot
1,normalisation,Normalise one typed English Grammar Profile de...
2,canonicalisation,grammar_cell_english_v1
3,generation,fixture_one_candidate_per_cell / controlled_pr...
4,validation,independent_item_judgment_v1
5,normalisation backend,fixture / deterministic
6,generation backend,fixture / deterministic
7,validation backend,fixture / deterministic
8,grammar fold,fixture_semantic_grammar_coverage_v1
9,simulation,structural_synthetic_world_v1 / development_ac...


## Typed EGP resource → normalisation

In [3]:
resources = load_typed_resource(RESOURCE_PATH, resource_schema)
display(pd.DataFrame(resources)[['source_id', 'subcategory', 'guideword', 'cefr']])

mappings = normalise(
    resources,
    phase1_prompt,
    phase2_prompt,
    normalisation_rulebook,
    grammar_schema,
    model=backend_settings['normalisation']['model'],
    reasoning_effort=backend_settings['normalisation']['reasoning_effort'],
    model_call=model_call,
    evidence_dir=WORK / 'normalisation',
)
display(pd.DataFrame([{
    'source_id': row['source_id'], 'result': row['result'],
    'cells': len(row['cells']), 'note': row['note'],
} for row in mappings]))
evidence = WORK / 'normalisation/calls/egp_present_simple_phase1/rendered_prompt.txt'
display(Markdown('**Rendered Phase-1 prompt excerpt**'))
print(evidence.read_text()[:700])

,source_id,subcategory,guideword,cefr
0,egp_present_simple,present simple,PRESENT SIMPLE AFFIRMATIVE,A2
1,egp_past_negative,past simple,PAST SIMPLE NEGATIVE,A2
2,egp_past_passive,passives,PAST SIMPLE PASSIVE,B1
3,egp_present_progressive,present continuous,PRESENT CONTINUOUS,A2
4,egp_progressive_passive_negative,passives,PRESENT CONTINUOUS PASSIVE NEGATIVE,B1
5,egp_should_question,should,SHOULD QUESTIONS,A2


,source_id,result,cells,note
0,egp_present_simple,complete,1,None
1,egp_past_negative,complete,1,None
2,egp_past_passive,complete,1,None
3,egp_present_progressive,complete,1,None
4,egp_progressive_passive_negative,complete,1,None
5,egp_should_question,complete,1,None


**Rendered Phase-1 prompt excerpt**

Normalise one typed English Grammar Profile descriptor.

Use only the declared Phase-1 evidence: source_id, supercategory, subcategory,
guideword, and can_do. Do not use examples or CEFR. Apply the supplied
canonical schema and EGP normalisation rulebook.

For each dimension, prefer an exact scalar; use separate cells for genuinely
asserted alternatives; use a list for a bounded but unresolved set; and use
null when the descriptor supplies no usable constraint. Do not invent a
Cartesian product.

Set `phase2_eligible` to the canonical dimensions that examples are allowed to
refine, in canonical order. Every named dimension must contain uncertainty in
at least one cell. Use an empty list when


## Canonicalisation → item generation

`GrammarCell + generation prompt + English generation rulebook + item design + item format → LLM candidates → independent validation → deterministic diverse bank selection`

> Vocabulary is model-selected but constrained to remain simple background material; vocabulary knowledge is not modeled in the KT simulation. The active method generates three independent candidates per cell and retains up to two valid variants (earliest valid, then maximum token-set distance). Fixture mode uses one candidate per cell so this offline walkthrough remains fast. Source IDs, KCs, learner evidence, and fold labels are outside generation input.

In [4]:
cells = canonicalise(mappings, grammar_schema)
display(pd.DataFrame([{'cell_id': cell['cell_id'], **cell['features']} for cell in cells]))

candidates = generate_items(
    cells,
    generation_prompt,
    generation_rulebook,
    generation_design,
    item_format,
    model=backend_settings['generation']['model'],
    reasoning_effort=backend_settings['generation']['reasoning_effort'],
    model_call=model_call,
    evidence_dir=WORK / 'generation',
)

example_call = WORK / 'generation/calls/candidate_cell_001_01'
print('INPUT GrammarCell')
display(cells[0])
print('GENERATION PROMPT')
print(generation_prompt)
print('RULEBOOK')
print(generation_rulebook)
print('DESIGN')
display(generation_design)
print('FORMAT')
display(item_format)
print('RENDERED LLM PROMPT')
print((example_call / 'rendered_prompt.txt').read_text())
print('MODEL OUTPUT')
display(json.loads((example_call / 'parsed_result.json').read_text()))
print('CandidateItem')
display(candidates[0])
display(pd.DataFrame(candidates)[['item_id', 'cell_id', 'prompt', 'target_answer']])

,cell_id,tense,aspect,voice,polarity,clause,modal
0,cell_001,present,none,active,positive,declarative,none
1,cell_002,past,none,active,negative,declarative,none
2,cell_003,past,none,passive,positive,declarative,none
3,cell_004,present,progressive,active,positive,declarative,none
4,cell_005,present,progressive,passive,negative,declarative,none
5,cell_006,NA,none,active,positive,polar_question,should


INPUT GrammarCell


{'cell_id': 'cell_001',
 'features': {'tense': 'present',
  'aspect': 'none',
  'voice': 'active',
  'polarity': 'positive',
  'clause': 'declarative',
  'modal': 'none'},
 'source_ids': ['egp_present_simple']}

GENERATION PROMPT
Create one controlled-production English exercise candidate for the supplied
target.

The GrammarCell is fixed. Do not choose, weaken, or reinterpret it. Realise
every target feature in the target answer, follow the item-format contract, and
choose your own lexical material and short context. Ordinary lexical content is
treated as known background knowledge, not as a learner skill. Use common,
concrete, high-frequency vocabulary that is unlikely to be the source of
learner difficulty. Avoid specialist terms, idioms, rare senses, obscure proper
nouns, culturally specific knowledge, and unnecessary lexical complexity.
Choose predicates and argument structures compatible with the fixed target
GrammarCell. Keep the context short, natural, pedagogically useful, and
answer-determinate, and vary lexical and contextual material naturally across
items.

`target_answer` must be the complete grammatical clause obtained when the
response slot is filled. Each string in `accepted_a

{'design_id': 'fixture_one_candidate_per_cell',
 'format': 'controlled_production',
 'generation': {'candidates_per_cell': 1,
  'candidate_calls': 'independent',
  'lexical_choice': 'model_generated',
  'lexical_constraint': 'simple_non_target_language',
  'nuisance_condition': 'short_context'},
 'bank_selection': {'maximum_items_per_cell': 1,
  'first_item': 'earliest_valid',
  'second_item': 'maximum_token_set_distance_from_first',
  'distance_text': 'prompt_and_target_answer',
  'tie_break': 'earliest_candidate'}}

...

RENDERED LLM PROMPT
Create one controlled-production English exercise candidate for the supplied
target.

The GrammarCell is fixed. Do not choose, weaken, or reinterpret it. Realise
every target feature in the target answer, follow the item-format contract, and
choose your own lexical material and short context. Ordinary lexical content is
treated as known background knowledge, not as a learner skill. Use common,
concrete, high-frequency vocabulary that is unlikely to be the source of
learner difficulty. Avoid specialist terms, idioms, rare senses, obscure proper
nouns, culturally specific knowledge, and unnecessary lexical complexity.
Choose predicates and argument structures compatible with the fixed target
GrammarCell. Keep the context short, natural, pedagogically useful, and
answer-determinate, and vary lexical and contextual material naturally across
items.

`target_answer` must be the complete grammatical clause obtained when the
response slot is filled. Each string in `accepted

{'prompt': 'Every morning, Lina ___ to work by bus. (travel)',
 'target_answer': 'Every morning, Lina travels to work by bus.',
 'accepted_answers': ['travels']}

CandidateItem


{'item_id': 'candidate_cell_001_01',
 'cell_id': 'cell_001',
 'format': 'controlled_production',
 'prompt': 'Every morning, Lina ___ to work by bus. (travel)',
 'target_answer': 'Every morning, Lina travels to work by bus.',
 'accepted_answers': ['travels'],
 'generation_metadata': {'candidate_index': 1,
  'candidate_count': 1,
  'model': 'fixture'}}

,item_id,cell_id,prompt,target_answer
0,candidate_cell_001_01,cell_001,"Every morning, Lina ___ to work by bus. (travel)","Every morning, Lina travels to work by bus."
1,candidate_cell_002_01,cell_002,"Yesterday, Omar ___ the door because it was st...","Yesterday, Omar did not open the door because ..."
2,candidate_cell_003_01,cell_003,"Yesterday, the meal ___ by Ana. (prepare)","Yesterday, the meal was prepared by Ana."
3,candidate_cell_004_01,cell_004,"Right now, Mia ___ a book. (read)","Right now, Mia is reading a book."
4,candidate_cell_005_01,cell_005,"Right now, the rooms ___ by the team. (not / c...","Right now, the rooms are not being cleaned by ..."
5,candidate_cell_006_01,cell_006,___ we take the earlier train? (should),Should we take the earlier train?


## Independent validation → fixed item bank

In [5]:
display(pd.DataFrame([
    {'criterion': name, **declaration}
    for name, declaration in validation_criteria['criteria'].items()
]))
validator_accepted, judgments = validate_items(
    candidates,
    cells,
    validation_prompt,
    validation_criteria,
    model=backend_settings['validation']['model'],
    reasoning_effort=backend_settings['validation']['reasoning_effort'],
    model_call=model_call,
    evidence_dir=WORK / 'validation',
)
accepted_items = select_item_bank(validator_accepted, generation_design)
item_bank_summary = bank_summary(
    candidates, validator_accepted, judgments, cells, selected_items=accepted_items
)
display(pd.DataFrame(judgments)[['item_id', 'accepted']])
print({
    **{key: item_bank_summary[key] for key in ('validator_accepted_candidates', 'validator_acceptance_rate', 'selected_bank_items', 'covered_cells')},
    'non_target_language_simplicity_pass_rate': item_bank_summary['criterion_pass_rates']['non_target_language_simplicity'],
})

,criterion,required,question
0,target_fidelity,True,Does the target answer express exactly the int...
1,grammaticality,True,Is the target answer grammatical English?
2,naturalness,True,Is the prompt-answer pair natural and coherent?
3,pedagogical_suitability,True,Is the item suitable for focused grammar pract...
4,determinacy,True,Does the prompt sufficiently determine the acc...
5,non_target_language_simplicity,True,Is the vocabulary and context sufficiently com...
6,no_answer_leakage,True,Does the prompt avoid displaying the completed...
7,no_extraneous_grammar,True,Can the item be solved without unrelated advan...
8,no_world_knowledge,True,Can the item be answered without external fact...


,item_id,accepted
0,candidate_cell_001_01,True
1,candidate_cell_002_01,True
2,candidate_cell_003_01,True
3,candidate_cell_004_01,True
4,candidate_cell_005_01,True
5,candidate_cell_006_01,True


{'validator_accepted_candidates': 6, 'validator_acceptance_rate': 1.0, 'selected_bank_items': 6, 'covered_cells': 6, 'non_target_language_simplicity_pass_rate': 1.0}


## Semantic grammar fold → development-derived KC candidates

The outcome-free fold is built from canonical feature tuples and fixed accepted-item support. Candidate construction then receives only development GrammarCells and fixed development item IDs/cell IDs. It derives feature-value, English-declared cell-deterministic operation, supported pairwise, and exact-development-cell hypotheses; support and activation equivalence are calculated before any learner outcome exists.

In [6]:
grammar_fold = build_semantic_fold(
    grammar_schema, cells, accepted_items, grammar_fold_design
)
display(pd.DataFrame(grammar_fold)[[
    'cell_id', 'grammar_split', 'accepted_item_support',
    'unseen_development_values', 'selection_reason',
]])

development_cell_ids = {
    row['cell_id'] for row in grammar_fold
    if row['grammar_split'] == 'development'
}
development_cells = [row for row in cells if row['cell_id'] in development_cell_ids]
development_items = [row for row in accepted_items if row['cell_id'] in development_cell_ids]
candidate_inventory = make_kc_candidates(
    grammar_schema, development_cells, development_items, candidate_design
)
candidate_rows = pd.DataFrame(candidate_inventory['candidates'])
display(candidate_rows[[
    'id', 'family', 'cell_support', 'item_support',
    'equivalent_to', 'selection_eligible',
]])
print('Selection candidate pool:', [
    row['id'] for row in candidate_inventory['candidates']
    if row['selection_eligible']
])

,cell_id,grammar_split,accepted_item_support,unseen_development_values,selection_reason
0,cell_003,development,1,[],development_acquisition
1,cell_002,development,1,[],development_acquisition
2,cell_001,compositional_holdout,1,[],semantic_sample_with_supported_development_con...
3,cell_005,development,1,[],development_acquisition
4,cell_006,novel_feature_holdout,1,"[{'dimension': 'tense', 'value': 'NA'}, {'dime...",contains_declared_novel_feature_value
5,cell_004,development,1,[],development_acquisition


,id,family,cell_support,item_support,equivalent_to,selection_eligible
0,kc_cell__tense_past__aspect_none__voice_active...,full_cell,1,1,kc_interaction__polarity_negative__and__tense_...,False
1,kc_cell__tense_past__aspect_none__voice_passiv...,full_cell,1,1,kc_interaction__tense_past__and__voice_passive,False
2,kc_cell__tense_present__aspect_progressive__vo...,full_cell,1,1,None,True
3,kc_cell__tense_present__aspect_progressive__vo...,full_cell,1,1,kc_interaction__aspect_progressive__and__polar...,False
4,kc_feature__aspect__progressive,feature_value,2,2,None,True
5,kc_feature__polarity__negative,feature_value,2,2,None,True
6,kc_feature__tense__past,feature_value,2,2,None,True
7,kc_feature__tense__present,feature_value,2,2,kc_feature__aspect__progressive,False
8,kc_feature__voice__passive,feature_value,2,2,None,True
9,kc_interaction__aspect_progressive__and__polar...,interaction,1,1,None,False


Selection candidate pool: ['kc_cell__tense_present__aspect_progressive__voice_active__polarity_positive__clause_declarative__modal_none', 'kc_feature__aspect__progressive', 'kc_feature__polarity__negative', 'kc_feature__tense__past', 'kc_feature__voice__passive', 'kc_operation__finite_tense_form']


## Development acquisition → frozen probes → evidence-based frozen policy → projection

Learners acquire development grammar only; one non-updating probe bank measures development, compositional, and novel-feature grammar at the same frozen learner state. The selector reads development acquisition evidence only, starts from reusable feature KCs, adds supported candidates under a validation-loss-plus-complexity objective, and freezes the resulting policy before projection. The notebook reduces only the learner count so the walkthrough stays quick.

In [7]:
simulation_world = materialize_latent_world(
    simulation_world_design, grammar_schema, cells
)
small_world = deepcopy(simulation_world)
small_world['learners'] = NOTEBOOK_LEARNERS
events = simulate_frozen_probes(
    accepted_items,
    grammar_fold,
    small_world,
    simulation_protocol,
    oracle_path=WORK / 'oracle_debug.json',
)
display(pd.DataFrame(events).head(8))
print({
    'learners': len({row['learner_id'] for row in events}),
    'acquisition_events': sum(row['protocol_phase'] == 'acquisition' for row in events),
    'probe_events': sum(row['protocol_phase'] == 'probe' for row in events),
    'events': len(events),
    'seed': small_world['seed'],
})

development_item_ids = {row['item_id'] for row in development_items}
development_events = [row for row in events if row['item_id'] in development_item_ids]
policy = select_kcs(candidate_inventory, development_events, selection_design)
display(pd.DataFrame(policy['kcs'])[['id', 'definition']])

projection = project_kcs(accepted_items, cells, policy)
display(pd.DataFrame(projection))

,event_id,learner_id,item_id,correct,sequence_index,dataset_split,item_difficulty,grammar_split,protocol_phase,updates_mastery,updates_history
0,event_0000001,learner_001,candidate_cell_002_01,0,1,train,0.19,development,acquisition,True,True
1,event_0000002,learner_001,candidate_cell_003_01,0,2,train,0.39,development,acquisition,True,True
2,event_0000003,learner_001,candidate_cell_004_01,1,3,train,-0.03,development,acquisition,True,True
3,event_0000004,learner_001,candidate_cell_005_01,0,4,train,0.61,development,acquisition,True,True
4,event_0000005,learner_001,candidate_cell_003_01,1,5,train,0.39,development,acquisition,True,True
5,event_0000006,learner_001,candidate_cell_004_01,1,6,train,-0.03,development,acquisition,True,True
6,event_0000007,learner_001,candidate_cell_005_01,1,7,train,0.61,development,acquisition,True,True
7,event_0000008,learner_001,candidate_cell_002_01,1,8,train,0.19,development,acquisition,True,True


{'learners': 8, 'acquisition_events': 160, 'probe_events': 48, 'events': 208, 'seed': 20260827}


,id,definition
0,kc_feature__aspect__progressive,Represent canonical aspect=progressive.
1,kc_feature__polarity__negative,Represent canonical polarity=negative.
2,kc_feature__tense__past,Represent canonical tense=past.
3,kc_feature__voice__passive,Represent canonical voice=passive.


,item_id,kc_ids
0,candidate_cell_001_01,[]
1,candidate_cell_002_01,"[kc_feature__polarity__negative, kc_feature__t..."
2,candidate_cell_003_01,"[kc_feature__tense__past, kc_feature__voice__p..."
3,candidate_cell_004_01,[kc_feature__aspect__progressive]
4,candidate_cell_005_01,"[kc_feature__aspect__progressive, kc_feature__..."
5,candidate_cell_006_01,[]


## Knowledge tracing → evaluation

In [8]:
predictions = run_kt(events, projection, kt_protocol)
display(pd.DataFrame(predictions).head(9))

results = evaluate(
    candidates,
    judgments,
    accepted_items,
    cells,
    grammar_fold,
    events,
    policy,
    projection,
    predictions,
    evaluation_protocol,
    validator_accepted_items=validator_accepted,
)
display(pd.DataFrame([
    {'technique': name, **{key: metrics[key] for key in ('n', 'log_loss', 'brier_score', 'auc')}}
    for name, metrics in results['kt'].items()
]))

,event_id,technique,probability,history_events
0,event_0000001,empirical,0.500000,0
1,event_0000002,empirical,0.416667,1
2,event_0000003,empirical,0.500000,2
3,event_0000004,empirical,0.444444,3
4,event_0000005,empirical,0.250000,4
5,event_0000006,empirical,0.500000,5
6,event_0000007,empirical,0.416667,6
7,event_0000008,empirical,0.400000,7
8,event_0000009,empirical,0.666667,8


,technique,n,log_loss,brier_score,auc
0,empirical,48,0.781880,0.292026,0.369489
1,bkt,48,0.773517,0.281976,0.515873
2,logistic,48,0.702968,0.254198,0.598765


## Executed object flow

In [9]:
WALKTHROUGH_SUMMARY = {
    'live_mode': LIVE_MODE,
    'source_descriptors': len(resources),
    'mappings': len(mappings),
    'canonical_cells': len(cells),
    'candidate_items': len(candidates),
    'accepted_items': len(accepted_items),
    'development_cells': sum(row['grammar_split'] == 'development' for row in grammar_fold),
    'compositional_holdout_cells': sum(row['grammar_split'] == 'compositional_holdout' for row in grammar_fold),
    'novel_feature_holdout_cells': sum(row['grammar_split'] == 'novel_feature_holdout' for row in grammar_fold),
    'learners': len({event['learner_id'] for event in events}),
    'acquisition_events': sum(event['protocol_phase'] == 'acquisition' for event in events),
    'probe_events': sum(event['protocol_phase'] == 'probe' for event in events),
    'events': len(events),
    'structural_candidates': candidate_inventory['candidate_counts']['raw_total'],
    'selection_eligible_candidates': candidate_inventory['candidate_counts']['selection_eligible'],
    'selected_kcs': len(policy['kcs']),
    'kt_techniques': kt_protocol['techniques'],
}
display(pd.DataFrame([
    ('Typed resource', len(resources)),
    ('Normalised mappings', len(mappings)),
    ('Canonical cells', len(cells)),
    ('Candidate items', len(candidates)),
    ('Accepted items', len(accepted_items)),
    ('Learner events', len(events)),
    ('Structural KC candidates', candidate_inventory['candidate_counts']['raw_total']),
    ('KC projections', len(projection)),
    ('KT predictions', len(predictions)),
], columns=['scientific object', 'records']))
print(json.dumps(WALKTHROUGH_SUMMARY, indent=2))

,scientific object,records
0,Typed resource,6
1,Normalised mappings,6
2,Canonical cells,6
3,Candidate items,6
4,Accepted items,6
5,Learner events,208
6,Structural KC candidates,27
7,KC projections,6
8,KT predictions,624


{
  "live_mode": false,
  "source_descriptors": 6,
  "mappings": 6,
  "canonical_cells": 6,
  "candidate_items": 6,
  "accepted_items": 6,
  "development_cells": 4,
  "compositional_holdout_cells": 1,
  "novel_feature_holdout_cells": 1,
  "learners": 8,
  "acquisition_events": 160,
  "probe_events": 48,
  "events": 208,
  "structural_candidates": 27,
  "selection_eligible_candidates": 6,
  "selected_kcs": 4,
  "kt_techniques": [
    "empirical",
    "bkt",
    "logistic"
  ]
}
